## Cruzamento dos Dados (Enunciado Questões + Microdados)

* **Input**: Local do CSV de Questões Extraídas + Local do CSV de Microdados

* **Output**: CSV com as colunas -> *numero_questao*, *enunciado*, *alternativas*, *nu_param_B* e *gabarito*

In [3]:
import pandas as pd

In [89]:
# Definição de Variáveis de Input e Output
flag_ledor=1
year = "2017"
if flag_ledor:
    year+="_LEDOR"

question_path = f"../data/extracted_questions/enem_{year}.csv"
microdados_path = f"../data/microdados/microdados_{year[:4]}.csv"
output_path = f"../data/merged_data/enem_{year}.csv"

In [90]:
# Ler o CSV extraído do PDF (resultado de Text_Extraction.ipynb)
df_questions = pd.read_csv(question_path, encoding="utf-8", quotechar='"')

# Ler o CSV de microdados
df_microdados = pd.read_csv(microdados_path, delimiter=";", encoding="latin1")

In [91]:
co_prova = {
    "2015": 275,
    "2016": 351,
    "2017": 391,
    "2017_LEDOR": 407,
    "2018_LEDOR": 463,
    "2019_LEDOR": 519,
    "2020": 597,
    "2020_LEDOR": 604,
    "2021_LEDOR": 916,
    "2022_LEDOR": 1092,
    "2023_LEDOR": 1228,
    "2009": 49,
    "2010": 89,
    "2011": 121
}

In [96]:
# Filtrando os microdados para SG_AREA 'CN', TX_COR 'AZUL' e CO_PROVA 'X'
# ATENÇÃO: O Código da Prova para a Aplicação Regular muda a cada ano
df_microdados_filtrado = df_microdados[
    (df_microdados["SG_AREA"] == "CN")  # Ciência das Naturezas
    & (
        (df_microdados["TX_COR"].str.upper() == "LARANJA" if flag_ledor else "AZUL") 
    )  # Apenas caderno azul
    & (df_microdados["CO_PROVA"] == co_prova[year])  # Aplicação Regular
]

# Selecionar apenas as colunas de interesse dos microdados
df_microdados_sel = df_microdados_filtrado[
    ["CO_POSICAO", "NU_PARAM_B", "TX_GABARITO"]
].copy()

if year == "2017_LEDOR":
    df_microdados_sel["CO_POSICAO"] += 90

1


In [97]:
# Certificar que os tipos das chaves de junção são compatíveis
df_questions["numero_questao"] = df_questions["numero_questao"].astype(str)
df_microdados_sel["CO_POSICAO"] = df_microdados_sel["CO_POSICAO"].astype(str)

# Fazer o cruzamento usando o número da questão (numero_questao e CO_POSICAO)
df_merged = pd.merge(
    df_questions,
    df_microdados_sel,
    left_on="numero_questao",
    right_on="CO_POSICAO",
    how="left",
)

In [99]:
# Remover a coluna CO_POSICAO (redundante)
df_merged.drop(columns=["CO_POSICAO"], inplace=True)

# Renomeando colunas
df_merged.rename(columns={
    "NU_PARAM_B": "nu_param_B",
    "TX_GABARITO": "gabarito"
}, inplace=True)


# Salvando DF resultante
df_merged.to_csv(output_path, index=False, encoding="utf-8")

print("Merge concluído.")

Merge concluído.


In [100]:
df = pd.read_csv(output_path)
df.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito
0,91,Um fato corriqueiro ao se cozinhar arroz é o \...,"A: reação do gás de cozinha com o sal, volatil...",1.35265,B
1,92,A classificação biológica proposta por Whittak...,A: tipos de células.; B: aspectos ecológicos.;...,2.03182,C
2,93,"Em uma colisão frontal entre dois automóveis, ...",A: um; B: dois; C: três; D: quatro; E: cinco,0.91191,B
3,94,Coagulação acelerada\nPesquisadores criaram um...,A: Filariose.; B: Hemofilia.; C: Aterosclerose...,0.69737,B
4,95,A farinha de linhaça dourada é um produto natu...,A: Destilado um.; B: Destilado dois.; C: Resíd...,2.60783,E
